# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys
import pprint

pp = pprint.PrettyPrinter(indent=4)

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from typing import List
from dotenv import load_dotenv
import chromadb
from tavily import TavilyClient
from lib.agents import Agent
from lib.llm import LLM
from lib.tooling import tool
from lib.memory import ShortTermMemory

In [3]:
# TODO: Load environment variables
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")
print(api_key)

voc-590237544168865462887669c7d0c41ed347.29711129


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
import chromadb


# Persistent client setup
# chroma_client = chromadb.PersistentClient(path="/workspace/Code/project/starter/chromadb")
# collection = chroma_client.get_collection("udaplay")

@tool(
    name="retrieve_game",
    description="Finds game industry results in the vector database using keyword-based search.",
)
def retrieve_game(query: str) -> list[dict]:
    """
    Retrieves game information from the database without using embedding-based similarity.
    It performs a keyword search to ensure results strictly contain the query terms.
    
    Args:
        query (str): The search term or game title to look for.
        
    Returns:
        list[dict]: Parsed game details including platform, publisher, name, and release year.
    """
    # FIX: Use .get() with where_document for keyword matching (no embeddings involved)
    # This bypasses the vector similarity search entirely.
    chroma_client = chromadb.PersistentClient(path="/workspace/code/project/starter/chromadb")
    collection = chroma_client.get_collection("udaplay")
    print('counnt', (collection.count()))
    results = collection.get(
        where_document={"$contains": query},
        limit=3,
        include=['documents']
    )

    games = []
    # FIX: collection.get() returns a flat list in 'documents'
    for doc in results.get('documents', []):
        try:
            # AI Engineer approach: Robust parsing with error handling
            game_info = {
                "platform": doc.split("platform: [")[1].split("]")[0],
                "publisher": doc.split("publisher: ")[1].split("\n")[0].strip(),
                "name": doc.split("name: ")[1].split("release date:")[0].strip(),
                "yearofrelease": doc.split("release date: (")[1].split(")")[0],
                "genre": doc.split("genre: ")[1].split("\n")[0].strip(),
                "description": doc.split("description: ")[1].strip(),
            }
            games.append(game_info)
        except (IndexError, AttributeError):
            # Skip malformed documents to prevent tool crashes
            continue
            
    return games

print(retrieve_game("What is a game for Nintendo Switch?") ) # Example usage
            

counnt 15
[]


#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

@tool(
    name="evaluate_retrieval",
    description="Evaluate the usability of retrieved documents for answering a question",
)
def evaluate_retrieval(question: str, retrieved_docs: List[str]) -> dict:
    """
    Evaluate the usability of retrieved documents for answering a question.

    Args:
        question (str): The original question from the user.
        retrieved_docs (List[str]): The list of retrieved documents most similar to the user query.

    Returns:
        dict: A dictionary containing evaluation results with keys 'useful' and 'description'.
    """
    llm = LLM(model="gpt-4o-mini", temperature=0.3)

    prompt = (
        "Your task is to evaluate if the documents are enough to respond to the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        "Retrieved Documents:\n" + "\n".join(retrieved_docs) + "\n\n"
        "Evaluate if these documents are useful to answer the question."
    )

    response = llm.invoke(prompt)

    # Assuming response is structured as follows:
    # {
    #     "useful": True/False,
    #     "description": "Detailed explanation of the evaluation"
    # }

    return response

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry.
# 
@tool(
    name="game_web_search",
    description="Search the web for game industry information",
)
def game_web_search(question: str) -> List[dict]:
    """
    Search the web for game industry information based on the question.

    Args:
        question (str): A question about the game industry.

    Returns:
        List[dict]: A list of dictionaries containing search results.
    """
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
    results = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    # Extract relevant information from the results
    formatted_results = {
        "answer": results.get("answer", ""),
        "results": results.get("results", [])
    }

    return formatted_results

game_web_search("What is the a game for Nintendo Switch?") 

{'answer': 'The best-known game for Nintendo Switch is The Legend of Zelda: Breath of the Wild. Super Mario Odyssey is also highly acclaimed. Metroid Prime Remastered is another popular title.',
 'results': [{'url': 'https://eliterev.wordpress.com/2025/05/30/my-top-25-favorite-nintendo-switch-games-of-all-time/',
   'title': 'My Top 25 Favorite Nintendo Switch Games Of All Time',
   'content': 'Super Smash Bros. Ultimate is far and away my most played game on Nintendo Switch. It truly is the culmination of 20 years of Sakurai’s work on the series, stuffed with more content than any reasonable person would have expected. I deleted it from my Switch’s hard drive a few weeks ago just because I was playing it mindlessly as a timewaster and felt like it was cutting in too much to other things I wanted to do. Smash Bros. will always be a game I do not think about playing, and- perhaps contributing to how I never found success in a competitive format- a game where I do not think while playing

### Agent

In [23]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
from lib.memory import ShortTermMemory

instructions=("""
    You are an AI Research Agent for the video game industry. "
    Your tasks include: "
    1) Answering questions using internal knowledge first. "
    2) Evaluating the sufficiency of this answer using the 'evaluate_retrieval' tool. "
    3) If the internal knowledge is insufficient, use the 'game_web_search' tool as a fallback. "
    4) You may use the 'retrieve_game' tool to fetch relevant documents from the vector database to support your answer. "
    
    Provide the final answer based on the best available source."
    Format your final output as follows:
    
    User question:
    <user's question>
    
    Final answer:
    <your full answer here>"
    
    Sources:
    - If based on local data, write: Local database
    - If based on web search, provide full clickable URLs like: 
      [Wikipedia - GameName](https://...)

    Tools used:
    - If no tools used, skip this section
    - If tools are used write them as bulletpoints e.g. 
      - evaluate_retrieval
      - game_web_search
      
    Make sure the response is factual, clear, and well-formatted.
    The final report must include your answer with citations (if any), and a list of tools used.
    """
)


agent = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=instructions,

)
agent.memory = ShortTermMemory()
result = agent.invoke("When was Halo Infinite and Silver released?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [24]:
import uuid
session_id = uuid.uuid4()
print(f"session_id = {session_id}")
result = agent.invoke("When was Halo Infinite  released?", session_id=session_id)

session_id = c69af675-4537-4f43-9bd9-9e3c3268f61f
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [25]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

result = agent.invoke("When was Halo Infinite released?",session_id=session_id)
print(result.get_final_state()["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
User question:
When was Halo Infinite released?

Final answer:
Halo Infinite was released on December 8, 2021.

Sources:
- Local database

Tools used:
- None


In [26]:
pp.pprint(result.get_final_state())

{   'current_tool_calls': None,
    'instructions': '\n'
                    '    You are an AI Research Agent for the video game '
                    'industry. "\n'
                    '    Your tasks include: "\n'
                    '    1) Answering questions using internal knowledge '
                    'first. "\n'
                    '    2) Evaluating the sufficiency of this answer using '
                    'the \'evaluate_retrieval\' tool. "\n'
                    '    3) If the internal knowledge is insufficient, use the '
                    '\'game_web_search\' tool as a fallback. "\n'
                    "    4) You may use the 'retrieve_game' tool to fetch "
                    'relevant documents from the vector database to support '
                    'your answer. "\n'
                    '\n'
                    '    Provide the final answer based on the best available '
                    'source."\n'
                    '    Format your final output as follow

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes